# NBA Injury Risk — Data Cleaning
## Injury Dataset
### Load & Filter by Date

In [ ]:
import pandas as pd

In [ ]:
dest = "../data/raw"
file = 'NBA Player Injury Stats(1951 - 2023).csv'

df_injury = pd.read_csv(f'{dest}/{file}')

# drop the junk index
df_injury = df_injury.drop(columns=["Unnamed: 0"])

In [ ]:
# filter to your window
df_injury["Date"] = pd.to_datetime(df_injury["Date"])
# df_injury = df_injury[df_injury["Date"].dt.year >= 2015]
df_injury = df_injury[df_injury["Date"] >= "2015-10-01"]
df_injury = df_injury.reset_index(drop=True)

print(f"Shape: {df_injury.shape}")
print(f"Date range: {df_injury['Date'].min()} → {df_injury['Date'].max()}")
print(f"\nNotes value counts (top 10):\n{df_injury['Notes'].value_counts().head(10)}")
print(f"\nNull counts:\n{df_injury.isnull().sum()}")
print(f"\nSample:\n{df_injury.head(10)}")

### Filter Non-Workload Injury Events

In [ ]:
# =============================================================================
# INJURY DATA FILTERING
# =============================================================================
# We filter the injury dataset to retain only IL placements that are
# plausibly driven by workload accumulation. The rationale for each
# decision is documented below.
#
# REMOVED:
#   - "activated from IL": return-to-play events, not injury occurrences
#   - "placed on IL for rest": load management decisions by the team,
#     not actual injuries. Including these would introduce false positives
#     since a healthy player being rested looks identical to an injured one
#     in the notes.
#   - "placed on IL with NBA health and safety protocols": COVID-19 related,
#     not driven by physical workload in any way.
#   - "placed on IL with illness": illness is not workload-driven and
#     cannot be predicted from game log features.
#   - Contact injuries (keywords: "contusion", "bruised", "lacerated",
#     "fracture", "concussion"): caused by player-to-player collisions, not cumulative
#     fatigue. Cohan et al. (2021) included contact injuries but noted
#     METIC performed worst on this category precisely because contact
#     injuries are essentially random events. Since our project is
#     explicitly workload-based, including them would add noise and hurt
#     what we're trying to predict.
#
# KEPT:
#   - Muscle injuries (strains, pulls, tears of hamstrings, calves, quads):
#     directly driven by fatigue and accumulated minutes — the most
#     workload-sensitive category.
#   - Ligament injuries (ACL, ankle sprains, knee sprains): among the most
#     common NBA injuries and strongly associated with accumulated load
#     and inadequate rest between games.
#   - Back injuries: chronic in nature and accumulate over a season,
#     particularly in high-minute players. Highly relevant to rolling
#     workload features.
#   - Foot injuries: the third most common injury site in the NBA and
#     often caused by stress from repeated high-impact activity.
#   - Core injuries (hip flexors, groin, abdominal strains): driven by
#     explosive cutting and directional changes that accumulate over
#     a heavy schedule.
#   - Upper body injuries (shoulders, elbows, wrists): can be overuse-
#     driven (e.g. rotator cuff tendinitis from shooting volume) and
#     are worth retaining, though they are less workload-sensitive than
#     lower body injuries.
#   - Knee injuries (sore knee, knee inflammation, etc.): retained
#     separately since knee notes don't always use ligament/muscle
#     language but are clearly load-related.
#   - Sore/tightness notes (e.g. "sore left ankle", "tightness in lower
#     back"): these are early warning signs of overuse and are exactly
#     what a workload model should be able to predict.
# =============================================================================

remove_keywords = [
    "activated from IL",
    "placed on IL for rest",
    "health and safety",
    "illness",
    "contusion",
    "bruised",
    "lacerated",
    "fracture",
    "concussion",
    "head cold",
    "headache",
    "sinus",
    "sore throat",
    "strep throat",
    "throat",
    "tooth",
    "dental",
    "respiratory",
    "dehydration",
    "nausea",
    "gastro",
    "conjunctivitis",
    "pink eye",
    "neuropathy",
    "COVID",
    "coronavirus",
    "quarantine",
    "not with team",
    "personal reasons",
    "suspension",
    "flu",
    "stomach virus",
    "upset stomach",
    "appendectomy",
    "pneumonia",
    "mononucleosis",
    "root canal",
    "oral surgery",
    "eye procedure",
    "kidney stone",
    "arrhythmia",
    "vestibular",
    "medical tests",
    "ineligible",
    "CBC",
    "conditioning"
]

### Apply Filters

In [ ]:
keep_mask = ~df_injury["Notes"].str.contains(
    "|".join(remove_keywords), case=False, na=False
)

# retain only IL placements (Relinquished populated)
placement_mask = df_injury["Relinquished"].notna()

df_injury_filtered = df_injury[keep_mask & placement_mask].reset_index(drop=True)

df_injury_filtered = df_injury_filtered[
    df_injury_filtered["Notes"] != "placed on IL (P)"
].reset_index(drop=True)

print(f"Original shape:  {df_injury.shape}")
print(f"Filtered shape:  {df_injury_filtered.shape}") # 6264
print(f"Removed:         {len(df_injury) - len(df_injury_filtered)} rows")
print(f"\nRemaining Notes (top 15):\n{df_injury_filtered['Notes'].value_counts().head(15)}")

In [ ]:
# print(df_injury_filtered["Notes"].value_counts().to_string())
df_injury_filtered.head(5)

### Checking Names to Merge with Player Game Logs.

In [ ]:
# print(df_injury_filtered["Relinquished"].value_counts().to_string())

In [ ]:
dtd_mask = df_injury_filtered["Relinquished"].str.contains("DTD", na=False)
df_injury_filtered[dtd_mask]

Three rows were identified where the `Relinquished` column contained injury descriptions instead of player names, indicating a CSV parsing error in the original dataset caused by a missing player name that shifted subsequent columns. Cross-referencing with Pro Sports Transactions confirmed that only one row could be tentatively linked to a player (Karl-Anthony Towns, 2020-12-27), but the match was not certain enough to assign. Since the player identity could not be reliably determined for any of the three rows, they were removed.

### Drop Malformed Rows

In [ ]:
df_injury_filtered = df_injury_filtered[~dtd_mask].reset_index(drop=True)

print(f"Filtered shape: {df_injury_filtered.shape}")

In [ ]:
# print(df_injury_filtered["Team"].value_counts().to_string())
# df_injury_filtered[df_injury_filtered["Relinquished"] == '76ers']
# 2018-12-27
# df_injury_filtered[df_injury_filtered["Date"] == "2018-12-27"]

### Map Team Names to Abbreviations

In [ ]:
team_name_to_abbrev = {
    "76ers": "PHI",
    "Bucks": "MIL",
    "Bulls": "CHI",
    "Cavaliers": "CLE",
    "Celtics": "BOS",
    "Clippers": "LAC",
    "Grizzlies": "MEM",
    "Hawks": "ATL",
    "Heat": "MIA",
    "Hornets": "CHA",
    "Jazz": "UTA",
    "Kings": "SAC",
    "Knicks": "NYK",
    "Lakers": "LAL",
    "Magic": "ORL",
    "Mavericks": "DAL",
    "Nets": "BKN",
    "Nuggets": "DEN",
    "Pacers": "IND",
    "Pelicans": "NOP",
    "Pistons": "DET",
    "Blazers": "POR",
    "Raptors": "TOR",
    "Rockets": "HOU",
    "Spurs": "SAS",
    "Suns": "PHX",
    "Thunder": "OKC",
    "Timberwolves": "MIN",
    "Warriors": "GSW",
    "Wizards": "WAS",
}

df_injury_filtered["TEAM_ABBREVIATION"] = df_injury_filtered["Team"].map(team_name_to_abbrev)

In [ ]:
# verify nothing was missed
print(df_injury_filtered["TEAM_ABBREVIATION"].isna().sum())

In [ ]:
df_injury_filtered.head()

In [ ]:
df_injury_filtered[df_injury_filtered["Relinquished"] == ' 76ers']
#df_injury_filtered[df_injury_filtered["Date"] == '2018-12-27']

### Inspect Player Name Issues

In [ ]:
# dual names
dual = df_injury_filtered["Relinquished"].str.contains("/", na=False)
print(f"Dual names: {dual.sum()}")
# print(df_injury_filtered[dual]["Relinquished"].value_counts().to_string())

In [ ]:
# parentheticals
paren = df_injury_filtered["Relinquished"].str.contains(r"\(", na=False)
print(f"\nParentheticals: {paren.sum()}")
print(df_injury_filtered[paren]["Relinquished"].value_counts().to_string())

In [ ]:
import re

def clean_player_name(name):
    if pd.isna(name):
        return name
    
    # take last segment if multiple slashes
    if "/" in name:
        name = name.split("/")[-1]
    
    # remove anything in parentheses
    name = re.sub(r"\(.*?\)", "", name)
    
    # strip extra whitespace
    name = name.strip()
    
    return name

In [ ]:
#df_injury_filtered["player_name_clean"] = df_injury_filtered["Relinquished"].apply(clean_player_name)

# spot check
# print(df_injury_filtered[dual | paren][["Relinquished", "player_name_clean"]].drop_duplicates().to_string())

In [ ]:
print(df_injury_filtered[dual | paren][["Relinquished"]].drop_duplicates().to_string())

In [ ]:
# df_injury_filtered[df_injury_filtered["Relinquished"].str.contains("Metta", na=False)]["Relinquished"].unique()

In [ ]:
# df_injury_filtered[df_injury_filtered["Relinquished"].str.contains("Nene", na=False)]["Relinquished"].unique()

### Apply Manual Name Overrides

##### There's leading space. We will use strip() to remove leading and trailing whitespace.

In [ ]:
manual_overrides = {
    # name corrections - applied to Relinquished before cleaning
    "Metta World Peace / Ron Artest": "Metta World Peace",
    "Nene / Nene Hilario / Maybyner Hilario": "Nene",
    "Milos Teodosic / Milos Tedosic": "Milos Teodosic",
    "Wesley Matthews / Wes Matthews Jr.": "Wesley Matthews",
    "Sheldon McClellan / Sheldon Mac": "Sheldon McClellan",
    "Zhou Qi / Qi Zhou": "Zhou Qi",
    "Nazareth Mitrou-Long / Naz Mitrou-Long / Naz Long": "Naz Mitrou-Long",
    "Moritz Wagner / Moe Wagner": "Moritz Wagner",
    "Maurice Harkless / Moe Harkless": "Maurice Harkless",
    "Jacob Wiley / Jake Wiley": "Jacob Wiley",
}

# remove leading and trailing whitespace
df_injury_filtered["Relinquished"] = df_injury_filtered["Relinquished"].str.strip()

df_injury_filtered["Relinquished"] = df_injury_filtered["Relinquished"].replace(manual_overrides)
df_injury_filtered["player_name_clean"] = df_injury_filtered["Relinquished"].apply(clean_player_name)

In [ ]:
print(df_injury_filtered[dual | paren][["Relinquished", "player_name_clean"]].drop_duplicates().to_string())

In [ ]:
# print(df_injury_filtered["player_name_clean"].value_counts().to_string())
# df_injury_filtered[df_injury_filtered["player_name_clean"] == "Sheldon McClellan / Sheldon Mac"]
# Relinquished 

##### We got a weird player name '76ers'. Only one row. We will remove this player

In [ ]:
df_injury_filtered[df_injury_filtered["Relinquished"] == '76ers']

In [ ]:
df_injury_filtered = df_injury_filtered[
    df_injury_filtered["player_name_clean"] != "76ers"
].reset_index(drop=True)

df_injury_filtered[df_injury_filtered["player_name_clean"] == '76ers']

## Pre-Match Sanity Checks

In [ ]:
# Null check on player_name_clean
print(df_injury_filtered["player_name_clean"].isna().sum())

# Empty strings after cleaning
print((df_injury_filtered["player_name_clean"] == "").sum())

# Verify the manual overrides worked
check_names = ["Metta World Peace", "Nene", "Milos Teodosic", "Wesley Matthews", "Sheldon McClellan", "Zhou Qi", "Naz Mitrou-Long"]
for name in check_names:
    count = (df_injury_filtered["player_name_clean"] == name).sum()
    print(f"{name}: {count}")

# Verify team mapping has no nulls
print(df_injury_filtered["TEAM_ABBREVIATION"].isna().sum())

## Fuzzy Match

In [ ]:
from rapidfuzz import process, fuzz

df_game_logs = pd.read_csv("../data/processed/player_game_logs_all.csv")

name_to_id = (
    df_game_logs[["PLAYER_ID", "PLAYER_NAME"]]
    .drop_duplicates()
    .set_index("PLAYER_NAME")["PLAYER_ID"]
    .to_dict()
)

print(f"Unique players in game logs: {len(name_to_id)}")

In [ ]:
def fuzzy_match_player(name, lookup, threshold=90):
    if pd.isna(name):
        return None, None, None
    
    match, score, _ = process.extractOne(
        name, lookup.keys(), scorer=fuzz.token_sort_ratio
    )
    
    if score >= threshold:
        return match, lookup[match], score
    return match, None, score  # return match but no ID if below threshold

In [ ]:
results = df_injury_filtered["player_name_clean"].apply(
    lambda x: pd.Series(fuzzy_match_player(x, name_to_id), index=["matched_name", "PLAYER_ID", "match_score"])
)

df_injury_filtered = pd.concat([df_injury_filtered, results], axis=1)

# inspect low confidence matches
low_confidence = df_injury_filtered[
    (df_injury_filtered["match_score"] < 90) | 
    (df_injury_filtered["PLAYER_ID"].isna())
]


In [ ]:
print(f"Low confidence or unmatched: {len(low_confidence)}")
print(low_confidence[["player_name_clean", "matched_name", "match_score"]].drop_duplicates().to_string())

In [ ]:
# df_injury_filtered[df_injury_filtered['player_name_clean'] == 'Chris Wilcox']
# 1179           Chris Wilcox           C.J. Wilcox    69.565217
# Don't match
df_injury_filtered[df_injury_filtered['player_name_clean'] == 'Eric Griffin']


In [ ]:
post_clean_overrides = {
    "Nikola Vucevic": "Nikola Vučević",
    "Jonas Valanciunas": "Jonas Valančiūnas",
    "Kristaps Porzingis": "Kristaps Porziņģis",
    "Luka Doncic": "Luka Dončić",
    "Vit Krejci": "Vít Krejčí",
    "Anzejs Pasecniks": "Anžejs Pasečņiks",
    "Vlatko Cancar": "Vlatko Cancar",  # check exact NBA API spelling
    "C.J. Miles": "CJ Miles",
    "J.R. Smith": "JR Smith",
    "J.T. Thor": "JT Thor",
    "Herb Jones": "Herbert Jones",
    "Marcus Morris": "Marcus Morris Sr.",
    "Jimmy Butler": "Jimmy Butler III",
    "Frank Mason": "Frank Mason III",
    "Reggie Bullock": "Reggie Bullock Jr.",
    "Naz Mitrou-Long": "Naz Mitrou-Long",  # verify NBA API spelling
}

df_injury_filtered["player_name_clean"] = df_injury_filtered["player_name_clean"].replace(post_clean_overrides)

In [ ]:
df_injury_filtered = df_injury_filtered.drop(columns=["matched_name", "PLAYER_ID", "match_score"])

In [ ]:
results = df_injury_filtered["player_name_clean"].apply(
    lambda x: pd.Series(fuzzy_match_player(x, name_to_id), index=["matched_name", "PLAYER_ID", "match_score"])
)

df_injury_filtered = pd.concat([df_injury_filtered, results], axis=1)

# inspect low confidence matches
low_confidence = df_injury_filtered[
    (df_injury_filtered["match_score"] < 90) | 
    (df_injury_filtered["PLAYER_ID"].isna())
]

In [ ]:
print(f"Low confidence or unmatched: {len(low_confidence)}")
print(low_confidence[["player_name_clean", "matched_name", "match_score"]].drop_duplicates().to_string())

In [ ]:
# df_injury_filtered[df_injury_filtered['player_name_clean'] == 'Vlatko Cancar']
post_clean_overrides = {
    "Mike Dunleavy Jr.": "Mike Dunleavy",
    "Tony Wroten Jr.": "Tony Wroten",
    "Gerald Henderson Jr.": "Gerald Henderson",
    "Ray McCallum Jr.": "Ray McCallum",
    "Wes Johnson": "Wesley Johnson",
    "George Papagiannis": "Georgios Papagiannis",
    "Stephen Zimmerman Jr.": "Stephen Zimmerman",
    "Mike Conley Jr.": "Mike Conley",
    "Wayne Selden Jr.": "Wayne Selden",
    "Nigel Hayes": "Nigel Hayes-Davis",
    "Matt Williams": "Matt Williams Jr.",
    "R.J. Nembhard Jr.": "Ruben Nembhard Jr.",
    "Sheldon McClellan": "Sheldon Mac",  # NBA API uses this name
    "James McAdoo": "James Michael McAdoo",  # verify NBA API spelling
    'Vlatko Cancar': 'Vlatko Čančar'
}

df_injury_filtered["player_name_clean"] = df_injury_filtered["player_name_clean"].replace(post_clean_overrides)

In [ ]:
df_injury_filtered = df_injury_filtered.drop(columns=["matched_name", "PLAYER_ID", "match_score"])

In [ ]:
results = df_injury_filtered["player_name_clean"].apply(
    lambda x: pd.Series(fuzzy_match_player(x, name_to_id), index=["matched_name", "PLAYER_ID", "match_score"])
)

df_injury_filtered = pd.concat([df_injury_filtered, results], axis=1)

# inspect low confidence matches
low_confidence = df_injury_filtered[
    (df_injury_filtered["match_score"] < 90) | 
    (df_injury_filtered["PLAYER_ID"].isna())
]

In [ ]:
print(f"Low confidence or unmatched: {len(low_confidence)}")
print(low_confidence[["player_name_clean", "matched_name", "match_score"]].drop_duplicates().to_string())

## Loading Game Logs
### Need to further clean Game Logs

In [ ]:
# load game logs
df_game_logs = pd.read_csv("../data/processed/player_game_logs_all.csv")

# drop irrelevant columns
rank_cols = [col for col in df_game_logs.columns if col.endswith("_RANK")]

drop_cols = rank_cols + [
    "NICKNAME",
    "NBA_FANTASY_PTS",
    "WNBA_FANTASY_PTS",
    "DD2",
    "TD3",
    "AVAILABLE_FLAG",
    "MIN_SEC",
    "TEAM_COUNT",
]

df_game_logs = df_game_logs.drop(columns=drop_cols)

# drop zero minute rows
df_game_logs = df_game_logs[df_game_logs["MIN"] > 0].reset_index(drop=True)

# fix dtypes
df_game_logs["GAME_DATE"] = pd.to_datetime(df_game_logs["GAME_DATE"])
df_injury_filtered["Date"] = pd.to_datetime(df_injury_filtered["Date"])
df_injury_filtered["PLAYER_ID"] = df_injury_filtered["PLAYER_ID"].astype("Int64")
df_game_logs["PLAYER_ID"] = df_game_logs["PLAYER_ID"].astype("Int64")

In [ ]:
print(f"Shape after cleaning: {df_game_logs.shape}")
print(f"Zero minute rows: {(df_game_logs['MIN'] == 0).sum()}")
print(f"Null MIN rows: {df_game_logs['MIN'].isna().sum()}")
print(f"Duplicate rows: {df_game_logs.duplicated().sum()}")

## Combine game logs with injury data

In [ ]:
# ### Combine game logs with injury data

def label_injury_windows(df_logs, df_injuries):
    df_logs = df_logs.copy()
    df_logs["injury_within_30_days"] = 0

    for player_id, group in df_injuries.groupby("PLAYER_ID"):
        dates = sorted(group["Date"].tolist())
        
        # merge overlapping 30-day windows
        merged_windows = []
        window_start = dates[0] - pd.Timedelta(days=30)
        window_end = dates[0]
        
        for date in dates[1:]:
            new_start = date - pd.Timedelta(days=30)
            if new_start <= window_end:
                window_end = max(window_end, date)
            else:
                merged_windows.append((window_start, window_end))
                window_start = new_start
                window_end = date
        
        merged_windows.append((window_start, window_end))
        
        for start, end in merged_windows:
            mask = (
                (df_logs["PLAYER_ID"] == player_id) &
                (df_logs["GAME_DATE"] >= start) &
                (df_logs["GAME_DATE"] <= end)
            )
            df_logs.loc[mask, "injury_within_30_days"] = 1
    
    return df_logs

In [ ]:
df_game_logs = label_injury_windows(df_game_logs, df_injury_filtered)

print(f"Total game log rows: {len(df_game_logs)}")
print(f"Injury labeled rows: {df_game_logs['injury_within_30_days'].sum()}")
print(f"Non-injury rows: {(df_game_logs['injury_within_30_days'] == 0).sum()}")
print(f"Injury rate: {df_game_logs['injury_within_30_days'].mean():.4f}")
print(df_game_logs.groupby("SEASON_YEAR")["injury_within_30_days"].mean())

We use a flexible injury-labeling system, keeping vague IL placements and notes on soreness or tightness, as these are early signs of overuse directly related to workload-based prediction. Although this leads to a higher positive class rate of 18% compared to stricter criteria, it more effectively captures the early injury signals our rolling window features are intended to identify.

Potential drawbacks in our methodology: 

1. Label noise: A player placed on IL on January 15th may have actually gotten injured on January 12th, January 10th, or even earlier. Labeling all games in the 30-day window as "pre-injury" but some of those early games may have been played when the player was completely healthy. This introduces noise into your positive class labels.
2. Underreporting and reporting delays: As Cohan himself acknowledges, teams are incentivized to obscure injuries for competitive advantage. The IL placement date may not reflect when the injury actually occurred, making the 30-day window an approximation at best. A player could have been playing hurt for weeks before being officially placed on IL.
3. Arbitrary window size: 30 days is not derived from any clinical or biomechanical evidence — it's a practical choice. Some injuries like ACL tears are acute and have no meaningful 30-day buildup, while others like stress fractures or tendinitis genuinely accumulate over months. A single window size treats all injury types the same way.
4. Overlapping windows: If a player gets injured twice within 30 days, their windows overlap and some games get labeled twice. This creates ambiguity in the label for those rows.
5. Healthy games mislabeled: A player who plays 25 games in 30 days before an injury had most of those games played while healthy. Labeling all 25 as pre-injury when only the last few may have been truly high-risk dilutes the signal. 
 
We adopt the 30-day lookback window following Cohan et al. (2021) to ensure methodological consistency and maximize positive class samples due to the severe class imbalance in the dataset, while recognizing that this introduces label noise since the exact injury occurrence is unknown.

In [ ]:
df_game_logs.head()

## Save cleaned data

In [ ]:
df_injury_filtered.to_csv("../data/processed/injury_data_cleaned.csv", index=False)
print("Saved: injury_data_cleaned.csv")

In [ ]:
df_game_logs.to_csv("../data/processed/game_logs_labeled.csv", index=False)
print("Saved: game_logs_labeled.csv")

## No need to run the cells below
#### This was to check if the injury rate was too high due to duplications

In [ ]:
# how many injury events per player
injury_counts = df_injury_filtered.groupby("PLAYER_ID").size().sort_values(ascending=False)
print(injury_counts.head(20))


In [ ]:
# check a specific high-injury player
top_player_id = injury_counts.index[0]
print(df_injury_filtered[df_injury_filtered["PLAYER_ID"] == top_player_id][["Date", "Notes", "player_name_clean"]])

In [ ]:
def label_injury_windows(df_logs, df_injuries):
    df_logs = df_logs.copy()
    df_logs["injury_within_30_days"] = 0

    for player_id, group in df_injuries.groupby("PLAYER_ID"):
        dates = sorted(group["Date"].tolist())
        
        # merge overlapping 30-day windows
        merged_windows = []
        window_start = dates[0] - pd.Timedelta(days=30)
        window_end = dates[0]
        
        for date in dates[1:]:
            new_start = date - pd.Timedelta(days=30)
            if new_start <= window_end:
                # overlapping — extend the window
                window_end = max(window_end, date)
            else:
                # no overlap — save current window and start new one
                merged_windows.append((window_start, window_end))
                window_start = new_start
                window_end = date
        
        merged_windows.append((window_start, window_end))
        
        # label game log rows
        for start, end in merged_windows:
            mask = (
                (df_logs["PLAYER_ID"] == player_id) &
                (df_logs["GAME_DATE"] >= start) &
                (df_logs["GAME_DATE"] <= end)
            )
            df_logs.loc[mask, "injury_within_30_days"] = 1
    
    return df_logs

In [ ]:
df_injury_filtered["PLAYER_ID"] = df_injury_filtered["PLAYER_ID"].astype("Int64")
df_game_logs["PLAYER_ID"] = df_game_logs["PLAYER_ID"].astype("Int64")

df_game_logs = label_injury_windows(df_game_logs, df_injury_filtered)

print(f"Injury rate: {df_game_logs['injury_within_30_days'].mean():.4f}")
print(df_game_logs["injury_within_30_days"].value_counts())

In [ ]:
# check if any players matched
test_player = 2738  # Andre Iguodala
test_injuries = df_injury_filtered[df_injury_filtered["PLAYER_ID"] == test_player]
test_logs = df_game_logs[df_game_logs["PLAYER_ID"] == test_player]

print(f"Iguodala injury events: {len(test_injuries)}")
print(f"Iguodala game log rows: {len(test_logs)}")
print(f"Iguodala labeled rows: {test_logs['injury_within_30_days'].sum()}")
print(f"\nInjury PLAYER_ID dtype: {df_injury_filtered['PLAYER_ID'].dtype}")
print(f"Game log PLAYER_ID dtype: {df_game_logs['PLAYER_ID'].dtype}")
print(f"\nSample injury IDs: {df_injury_filtered['PLAYER_ID'].head()}")
print(f"Sample game log IDs: {df_game_logs['PLAYER_ID'].head()}")

In [ ]:
injury_by_season = df_game_logs.groupby("SEASON_YEAR")["injury_within_30_days"].mean()
print(injury_by_season)

In [ ]:
non_covid = df_game_logs[~df_game_logs["SEASON_YEAR"].isin(["2019-20", "2020-21"])]
print(f"Injury rate without COVID seasons: {non_covid['injury_within_30_days'].mean():.4f}")

In [ ]:
vague_keywords = ["placed on IL$", "sore", "tightness"]

vague_mask = df_injury_filtered["Notes"].str.contains(
    "|".join(vague_keywords), case=False, na=False, regex=True
)

print(f"Rows that would be removed: {vague_mask.sum()}")
print(f"Remaining injury events: {(~vague_mask).sum()}")

In [ ]:
# test on a copy
df_injury_strict = df_injury_filtered[~vague_mask].copy()
df_game_logs_strict = label_injury_windows(df_game_logs.drop(columns="injury_within_30_days"), df_injury_strict)

print(f"\nStrict injury rate: {df_game_logs_strict['injury_within_30_days'].mean():.4f}")
print(df_game_logs_strict.groupby("SEASON_YEAR")["injury_within_30_days"].mean())